In [70]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [71]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader

In [72]:
device=torch.device('cuda' if torch.cuda.is_available()
else 'cpu')
device

device(type='cpu')

In [73]:
corpus = [
    ("Ami ranna korte bhalo bashi", "positive"),
    ("Ei khabarer shadh khub e baje", "negative"),
    ("Ajker din ta onek shundor chilo", "positive"),
    ("Kajta ekdom e pochondo hoyni", "negative"),
    ("Boita pore amar khub bhalo laglo", "positive"),
    ("Emon ghorami amar ekdom bhalo lage na", "negative")
]

In [74]:
import re
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)
    return text.split()
t='ami bekar ,LIfe a chiLL drkr!!'
clean_text(t)

['ami', 'bekar', 'life', 'a', 'chill', 'drkr']

In [75]:
# build vocabulary
all_words=[]
for text,label in corpus:
  all_words.extend(clean_text(text))
print(all_words)

['ami', 'ranna', 'korte', 'bhalo', 'bashi', 'ei', 'khabarer', 'shadh', 'khub', 'e', 'baje', 'ajker', 'din', 'ta', 'onek', 'shundor', 'chilo', 'kajta', 'ekdom', 'e', 'pochondo', 'hoyni', 'boita', 'pore', 'amar', 'khub', 'bhalo', 'laglo', 'emon', 'ghorami', 'amar', 'ekdom', 'bhalo', 'lage', 'na']


In [76]:
# padd token=0, unknown_token=1
vocab={'<PAD>':0,
       '<UNK>':1}
from collections import Counter
words_couter=Counter(all_words)
for word,count in words_couter.items():
  # print(word,count)
  vocab[word]=len(vocab)

print(vocab)


{'<PAD>': 0, '<UNK>': 1, 'ami': 2, 'ranna': 3, 'korte': 4, 'bhalo': 5, 'bashi': 6, 'ei': 7, 'khabarer': 8, 'shadh': 9, 'khub': 10, 'e': 11, 'baje': 12, 'ajker': 13, 'din': 14, 'ta': 15, 'onek': 16, 'shundor': 17, 'chilo': 18, 'kajta': 19, 'ekdom': 20, 'pochondo': 21, 'hoyni': 22, 'boita': 23, 'pore': 24, 'amar': 25, 'laglo': 26, 'emon': 27, 'ghorami': 28, 'lage': 29, 'na': 30}


In [77]:
label_map={'negative':0,'positive':1}
idx_to_label={v:k for k,v in label_map.items()}
print(idx_to_label)

{0: 'negative', 1: 'positive'}


In [78]:
# hyperparameters
Max_Len=6
Embedding_dim=8
Hidden_dim=16
output_dim=2
Batch_size=2
learning_rate=0.05
epochs=10

#Dataset Class

In [79]:
class SentimentDataset(Dataset):
  def __init__(self,corpus,vocab,label_map):
    self.data=[]

    for text,label in corpus:
      tokens=clean_text(text)
      token_ids=[vocab.get(token,vocab['<UNK>']) for token in tokens]

      label_id=label_map[label]
      self.data.append((token_ids,label_id))


  def __len__(self):
    return len(self.data)

  def __getitem__(self,idx):
    return self.data[idx]





```
DataLoader- helper function *collate_fn*

let's say ,batch size=2
  
batch

[
([2,3,4,5,6,7],1),

([3,8,9,10],0)
]

প্রথমে

sequences=[]
labels=[]
sequences

[]

labels

[]

Loop

for token_ids,label_id in batch

প্রথম iteration

token_ids

[2,3,4,5,6,7]

label

1

ধরো

MAX_LEN=8

Length

6

Condition

if len(token_ids)<MAX_LEN
6<8

True

Padding

token_ids=
token_ids+
[vocab["<PAD>"]]*
(MAX_LEN-len(token_ids))

PAD

0

তাহলে

[2,3,4,5,6,7]

+

[0,0]

Output

[2,3,4,5,6,7,0,0]

এরপর

sequences

[
[2,3,4,5,6,7,0,0]
]

labels

[1]

দ্বিতীয় iteration

token_ids

[3,8,9,10]

Length
4
Padding

[3,8,9,10,0,0,0,0]

এখন

sequences

[
[2,3,4,5,6,7,0,0],
[3,8,9,10,0,0,0,0]
]

labels
[1,0]
```





```
Raw Corpus
─────────────────────────────────────
("আমি এই সিনেমাটি খুব পছন্দ করেছি", "positive")
("এই সিনেমা একদম বাজে", "negative")
             │
             ▼
clean_text()
             │
             ▼
["আমি","এই","সিনেমাটি","খুব","পছন্দ","করেছি"]
["এই","সিনেমা","একদম","বাজে"]
             │
             ▼
Vocabulary Mapping
             │
             ▼
[2,3,4,5,6,7]
[3,8,9,10]
             │
             ▼
SentimentDataset
             │
             ▼
[
 ([2,3,4,5,6,7], 1),
 ([3,8,9,10], 0)
]
             │
             ▼
DataLoader (batch_size=2)
             │
             ▼
sentiment_collate_fn()
             │
             ├── Padding / Truncation
             ▼
[
 [2,3,4,5,6,7,0,0],
 [3,8,9,10,0,0,0,0]
]
             │
             ▼
torch.tensor(...)
             │
             ▼
Input Tensor Shape = (2, 8)
Label Tensor Shape = (2,)
             │
             ▼
Embedding Layer → LSTM/GRU/Transformer → Classifier → Prediction
```



In [80]:
def sentiment_collate_fn(batch):
  sequences=[]
  labels=[]

  for token_ids,label_id in batch:

    if len(token_ids)<Max_Len:
      token_ids=token_ids+[vocab['<PAD>']]*(Max_Len-len(token_ids))
    else :
      token_ids=token_ids[:Max_Len]

    sequences.append(token_ids)
    labels.append(label_id)

  return torch.tensor(sequences,dtype=torch.long),torch.tensor(labels,dtype=torch.long)
  # (2,8)


#dataLoader

In [81]:
dataset=SentimentDataset(corpus,vocab,label_map)
dataloader=DataLoader(dataset,batch_size=Batch_size,collate_fn=sentiment_collate_fn)

#Model

In [82]:
class sentimentClassifier(nn.Module):
  def __init__(self,vocab_size,embedding_dim,hidden_dim,output_dim):
    super(sentimentClassifier,self).__init__()
    self.embedding=nn.Embedding(num_embeddings=vocab_size,
                                 embedding_dim=embedding_dim)
    self.fc1=nn.Linear(embedding_dim,hidden_dim)
    self.relu=nn.ReLU()
    self.fc2=nn.Linear(hidden_dim,output_dim)


  def forward(self,x):
    embedded=self.embedding(x) #(batch_size,MaxLen,embedding_dim)

    pooled=torch.mean(embedded,dim=1) #(batch_size,embeddding_dim)

    out=self.fc1(pooled)
    out=self.relu(out)
    logits=self.fc2(out)
    return logits




In [83]:
model=sentimentClassifier(vocab_size=len(vocab),
                          embedding_dim=Embedding_dim,
                          hidden_dim=Hidden_dim,output_dim=output_dim).to(device)
model

sentimentClassifier(
  (embedding): Embedding(31, 8)
  (fc1): Linear(in_features=8, out_features=16, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=16, out_features=2, bias=True)
)

In [84]:
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=learning_rate)

#Model Train

In [85]:
model.train()

for epoch in range(epochs):
  epoch_loss=0
  correct=0
  total=0

  for texts,labels in dataloader:
    texts,labels=texts.to(device),labels.to(device)

    output=model(texts)
    loss=criterion(output,labels)
    # print(loss)
    # # print("="*100)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    epoch_loss+=loss.item()
    _,predicted=torch.max(output,dim=1)
    total+=labels.size(0)
    correct+=(predicted==labels).sum().item()

  print(f"Epoch [{epoch+1}/{epochs}], Loss: {epoch_loss/len(dataloader):.4f}, Accuracy: {(correct/total)*100:.2f}%")

Epoch [1/10], Loss: 0.6217, Accuracy: 66.67%
Epoch [2/10], Loss: 0.3747, Accuracy: 100.00%
Epoch [3/10], Loss: 0.1297, Accuracy: 100.00%
Epoch [4/10], Loss: 0.0166, Accuracy: 100.00%
Epoch [5/10], Loss: 0.0012, Accuracy: 100.00%
Epoch [6/10], Loss: 0.0001, Accuracy: 100.00%
Epoch [7/10], Loss: 0.0000, Accuracy: 100.00%
Epoch [8/10], Loss: 0.0000, Accuracy: 100.00%
Epoch [9/10], Loss: 0.0000, Accuracy: 100.00%
Epoch [10/10], Loss: 0.0000, Accuracy: 100.00%


#Predictive System

In [90]:
def predict_sentiment(text,model,vocab,label_map,Max_Len):
  model.eval()

  with torch.no_grad():
    tokens=clean_text(text)
    token_ids=[vocab.get(token,vocab['<UNK>']) for token in tokens]

    if len(token_ids)<Max_Len:
      token_ids=token_ids+[vocab['<PAD>']]*(Max_Len-len(token_ids))
    else:
      token_ids=token_ids[:Max_Len]

    print("token_ids",token_ids)

    input_tensor=torch.tensor([token_ids],dtype=torch.long).to(device)

    logits=model(input_tensor)

    probabilities=torch.softmax(logits,dim=1)
    predicted_class=torch.argmax(probabilities,dim=1).item()

    confidence, predicted_idx = torch.max(probabilities, dim=1)
    prediction = idx_to_label[predicted_idx.item()]

    print(f"\nইনপুট টেক্সট: '{text}'")
    print(f"মডেলের প্রেডিকশন: {prediction.upper()} (নিশ্চয়তা বা Confidence: {confidence.item()*100:.2f}%)")



In [91]:
predict_sentiment("Amar khub bhalo laglo", model, vocab, label_map, Max_Len)

token_ids [25, 10, 5, 26, 0, 0]

ইনপুট টেক্সট: 'Amar khub bhalo laglo'
মডেলের প্রেডিকশন: POSITIVE (নিশ্চয়তা বা Confidence: 100.00%)




---

###

মডেলের ভেতরে ডেটা কীভাবে রূপান্তরিত হচ্ছে তা একটি উদাহরণের মাধ্যমে নিচে ডাইমেনশন বা শেপ (Shape) সহ ভিজ্যুয়ালাইজ করা হলো।

ধরে নেই, আমাদের কাছে ২ আকারের একটি ব্যাচ (`BATCH_SIZE = 2`) আছে এবং বাক্যের সর্বোচ্চ দৈর্ঘ্য ৬ (`MAX_LEN = 6`)।

* বাক্য ১: `"Amar khub bhalo laglo"` $\rightarrow$ টোকেন আইডি: `[12, 15, 6, 13, 0, 0]` (প্যাডিংসহ)
* বাক্য ২: `"Ei khabarer shadh baje"` $\rightarrow$ টোকেন আইডি: `[3, 4, 5, 8, 0, 0]` (প্যাডিংসহ)

#### ধাপ ১: ইনপুট টেনসর (Input Tensor)

`DataLoader` থেকে যখন এই ব্যাচটি বের হবে, তখন তার ম্যাট্রিক্স রূপ হবে এমন:

$$\text{ইনপুট (x)} = \begin{bmatrix} 12 & 15 & 6 & 13 & 0 & 0 \\ 3 & 4 & 5 & 8 & 0 & 0 \end{bmatrix}$$

* **ডাইমেনশন বা শেপ:** `[Batch_Size, Max_Len]` $\rightarrow$ **`[2, 6]`**

---

#### ধাপ ২: এমবেডিং লেয়ার (Embedding Layer)

মডেলের `self.embedding` লেয়ারে প্রতিটি শব্দের আইডির বিপরীতে একটি ৮-ডাইমেনশনের ভেক্টর (`EMBEDDING_DIM = 8`) তৈরি হয়।
সহজ কথায়, ১টি সংখ্যা এখন ৮টি দশমিক সংখ্যায় রূপান্তরিত হবে।

* **ইনপুট শেপ:** `[2, 6]`
* **রূপান্তর:** প্রতিটি টোকেন আইডি $\rightarrow$ `[e1, e2, e3, e4, e5, e6, e7, e8]`
* **আউটপুট শেপ:** `[Batch_Size, Max_Len, Embedding_Dim]` $\rightarrow$ **`[2, 6, 8]`**

---

#### ধাপ ৩: গ্লোবাল অ্যাভারেজ পুলিং (Global Average Pooling)

কোডের এই লাইনটি লক্ষ্য করুন: `pooled = torch.mean(embedded, dim=1)`। এটি বাক্যের দৈর্ঘ্য বরাবর (`dim=1`) সব শব্দের ভেক্টরগুলোর গড় (Average) বের করে। এর ফলে পুরো বাক্যের অর্থ একটি একক ভেক্টরে চলে আসে।

* **ইনপুট শেপ:** `[2, 6, 8]` (২টি বাক্য, যার প্রতিটিতে ৬টি শব্দ আছে এবং প্রতিটি শব্দের ৮টি করে মান আছে)
* **কাজ:** ৬টি শব্দের ভেক্টর যোগ করে ৬ দিয়ে ভাগ করা হয়।
* **আউটপুট শেপ:** `[Batch_Size, Embedding_Dim]` $\rightarrow$ **`[2, 8]`**
* **ভিজ্যুয়ালাইজেশন:**

$$\text{pooled} = \begin{bmatrix} [\text{বাক্য ১ এর গড় ভেক্টর — ৮টি মান}] \\ [\text{বাক্য ২ এর গড় ভেক্টর — ৮টি মান}] \end{bmatrix}$$



---

#### ধাপ ৪: প্রথম লিনিয়ার লেয়ার এবং অ্যাক্টিভেশন (`fc1` + `ReLU`)

কোড: `out = self.fc1(pooled)` এবং `out = self.relu(out)`

এখানে মডেল তার ভেতরের ওয়েট (Weights) দিয়ে বাক্যের বৈশিষ্ট্যগুলো বোঝার চেষ্টা করে এবং নেতিবাচক মানগুলোকে `ReLU` দিয়ে শূন্য (0) বানিয়ে দেয়।

* **ইনপুট শেপ:** `[2, 8]`
* **লেয়ারের ওয়েট শেপ:** `[Embedding_Dim, Hidden_Dim]` $\rightarrow$ `[8, 16]`
* **আউটপুট শেপ:** `[Batch_Size, Hidden_Dim]` $\rightarrow$ **`[2, 16]`**

---

#### ধাপ ৫: আউটপুট লিনিয়ার লেয়ার বা লজিটস (`fc2`)

কোড: `logits = self.fc2(out)`

এটি আমাদের ফাইনাল লেয়ার। এটি ১৬ ডাইমেনশনের ফিচারকে আমাদের ক্লাসের সংখ্যায় (`OUTPUT_DIM = 2`, অর্থাৎ নেগেটিভ বা পজিটিভ) রূপান্তর করে। এই কাঁচা স্কোরগুলোকে বলা হয় **Logits**।

* **ইনপুট শেপ:** `[2, 16]`
* **আউটপুট শেপ:** `[Batch_Size, Output_Dim]` $\rightarrow$ **`[2, 2]`**
* **ভিজ্যুয়ালাইজেশন (কাল্পনিক আউটপুট):**

$$\text{logits} = \begin{bmatrix} -1.20 & 3.50 \\ 2.80 & -0.50 \end{bmatrix} \begin{matrix} \leftarrow \text{বাক্য ১ এর স্কোর [Negative, Positive]} \\ \leftarrow \text{বাক্য ২ এর স্কোর [Negative, Positive]} \end{matrix}$$



---

### ৩. কীভাবে মডেল প্রেডিকশন বা সিদ্ধান্ত নেয়?

ইনফারেন্স বা প্রেডিকশন ফাংশনে এই `Logits` বা স্কোরগুলোকে আসল সম্ভাবনায় (Probability) রূপান্তর করতে আমরা `torch.softmax` ব্যবহার করি।

ধরি, টেস্ট বাক্য `"Amar khub bhalo laglo"` এর জন্য মডেলের ফাইনাল লজিটস আউটপুট এসেছে: `[-1.20, 3.50]`।

১. **সফটম্যাক্স (Softmax) হিসাব:**
সফটম্যাক্স ফাংশন এই মানগুলোকে এমনভাবে রূপান্তর করে যেন তাদের যোগফল ১ বা ১০০% হয়।

* Class 0 (Negative) এর সম্ভাবনা: $e^{-1.20} / (e^{-1.20} + e^{3.50}) = 0.009$ (অর্থাৎ **০.৯%**)
* Class 1 (Positive) এর সম্ভাবনা: $e^{3.50} / (e^{-1.20} + e^{3.50}) = 0.991$ (অর্থাৎ **৯৯.১%**)

২. **আউটপুট ভেক্টর:** `[0.009, 0.991]`

৩. **সিদ্ধান্ত গ্রহণ (`torch.max`):**
`torch.max(probabilities, dim=1)` ফাংশনটি সবচেয়ে বড় মান এবং তার ইনডেক্স খুঁজে বের করে। এখানে সবচেয়ে বড় মান `0.991` যা ইনডেক্স `1`-এ অবস্থিত।

৪. **ম্যাপিং:** আমাদের `idx_to_label` ডিকশনারি অনুযায়ী ইনডেক্স `1` মানে হচ্ছে **"POSITIVE"**।

মডেলটি তখন স্ক্রিনে প্রিন্ট করবে: `PREDICTION: POSITIVE (Confidence: 99.1%)`।
```

